# 🚗 AI-Based Engine Operating Condition Classification with an MLP

Seven current sensor readings → MLP → Normal, High Load, Overheating, Inefficient Operation, or Abnormal.

> Recommendations are simulated educational responses.

👉 **Open the interactive companion:** [https://engine-operating-condition.streamlit.app](https://engine-operating-condition.streamlit.app/?stage=start)

## Complete workflow

Generate/load rows → clean → normalize → train MLP → enter readings → condition probabilities → simulated response.

## Interactive learning journey

- [Seven Gauges, One Condition](https://engine-operating-condition.streamlit.app/?stage=problem) — Operating-State Classification
- [The Seven Engine Measurements](https://engine-operating-condition.streamlit.app/?stage=inputs) — Input Features
- [Engine Operating Regimes](https://engine-operating-condition.streamlit.app/?stage=data) — Synthetic Labelled Dataset
- [Checking and Scaling the Log](https://engine-operating-condition.streamlit.app/?stage=prepare) — Cleaning and Normalization
- [Choosing the Right Neural Network](https://engine-operating-condition.streamlit.app/?stage=mlp) — Multilayer Perceptron
- [Learning Combined Operating Patterns](https://engine-operating-condition.streamlit.app/?stage=training) — Supervised Classification
- [Current Engine Condition](https://engine-operating-condition.streamlit.app/?stage=prediction) — Softmax Probabilities
- [The Engine Monitoring Audit](https://engine-operating-condition.streamlit.app/?stage=audit) — Confusion Matrix and Confidence Review

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report,confusion_matrix,ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input,Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping
SEED=42;np.random.seed(SEED);tf.random.set_seed(SEED)
FEATURES=["rpm","load_pct","coolant_c","intake_c","fuel_l_hr","manifold_bar","exhaust_c"]
CLASSES=["NORMAL","HIGH LOAD","OVERHEATING","INEFFICIENT","ABNORMAL"]

---
# 1. Seven Gauges, One Condition
### Phase 1 of 6 · The Engine Snapshot

## Part 1 · At the engine
An engine operator sees speed, load, cooling, intake, fuel, pressure, and exhaust measurements at the same moment.

## Part 2 · The engineering challenge
One value can appear acceptable while the combination indicates overheating, inefficient combustion, overload, or another abnormal state.

## Part 3 · Where the AI comes in
Learn the joint relationship among seven readings and return one current operating-condition probability distribution.

**Mechanical Engineering:** Seven Gauges, One Condition → **AI:** Operating-State Classification → `combine readings into one condition`

> 🎬 **See this illustrated and interactive:** [https://engine-operating-condition.streamlit.app/?stage=problem](https://engine-operating-condition.streamlit.app/?stage=problem)

## Part 4 · The technical explanation

Each sample is an independent snapshot. The notebook deliberately uses no sequence window: it asks what condition the engine is in now from seven synchronized values.

## Part 5 · What you just built

**In the notebook:** Define the five conditions and seven-input snapshot.

**Takeaway:** The condition exists in the combination, not necessarily in one gauge.

[Project overview](https://engine-operating-condition.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: The Seven Engine Measurements](https://engine-operating-condition.streamlit.app/?stage=inputs) ▶

---
# 2. The Seven Engine Measurements
### Phase 1 of 6 · The Engine Snapshot

## Part 1 · At the engine
Each sensor describes a different part of engine operation: mechanical demand, thermal state, airflow, fuelling, intake pressure, and combustion temperature.

## Part 2 · The engineering challenge
The features use different units and ranges, so numerical magnitude is not the same as engineering importance.

## Part 3 · Where the AI comes in
Keep all seven named columns and preserve their units for interpretation before scaling copies for the network.

**Mechanical Engineering:** The Seven Engine Measurements → **AI:** Input Features → `RPM, load, coolant, intake, fuel, pressure, exhaust`

> 🎬 **See this illustrated and interactive:** [https://engine-operating-condition.streamlit.app/?stage=inputs](https://engine-operating-condition.streamlit.app/?stage=inputs)

## Part 4 · The technical explanation

In [ ]:
example=dict(rpm=2400,load_pct=78,coolant_c=96,intake_c=42,fuel_l_hr=8.4,manifold_bar=1.3,exhaust_c=520)
pd.Series(example,name="Example sensor snapshot")

## Part 5 · What you just built

**In the notebook:** Create the feature list and inspect representative rows.

**Takeaway:** Named engineering features make the MLP input explainable.

◀ [Previous: Seven Gauges, One Condition](https://engine-operating-condition.streamlit.app/?stage=problem) &nbsp;|&nbsp; [Project overview](https://engine-operating-condition.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Engine Operating Regimes](https://engine-operating-condition.streamlit.app/?stage=data) ▶

---
# 3. Engine Operating Regimes
### Phase 2 of 6 · Preparing Sensor Records

## Part 1 · At the engine
Controlled tests label examples as Normal, High Load, Overheating, Inefficient Operation, or Abnormal.

## Part 2 · The engineering challenge
Purely random labels would let the notebook run but teach no meaningful engineering relationship.

## Part 3 · Where the AI comes in
Generate overlapping regimes from reasonable educational ranges and include noise so no single threshold solves every row.

**Mechanical Engineering:** Engine Operating Regimes → **AI:** Synthetic Labelled Dataset → `five physically distinguishable distributions`

> 🎬 **See this illustrated and interactive:** [https://engine-operating-condition.streamlit.app/?stage=data](https://engine-operating-condition.streamlit.app/?stage=data)

## Part 4 · The technical explanation

In [ ]:
def make_class(label,n=600,seed=0):
 rng=np.random.default_rng(seed)
 centres=[[1600,42,82,30,4.8,1.0,370],[2850,90,91,39,9.2,1.48,545],[2250,68,111,45,8.0,1.25,590],[1950,55,89,36,9.1,.98,475],[2100,58,97,43,7.8,1.12,535]][label]
 scales=[420,12,5,5,.8,.12,45] if label<4 else [800,24,13,11,2.1,.28,110]
 a=rng.normal(centres,scales,(n,7));a[:,0]=np.clip(a[:,0],600,4200);a[:,1]=np.clip(a[:,1],0,100);a[:,2]=np.clip(a[:,2],50,125);a[:,3]=np.clip(a[:,3],5,70);a[:,4]=np.clip(a[:,4],1,14);a[:,5]=np.clip(a[:,5],.5,2);a[:,6]=np.clip(a[:,6],180,750)
 return pd.DataFrame(a,columns=FEATURES).assign(condition=label)
data=pd.concat([make_class(i,seed=100+i) for i in range(5)],ignore_index=True).sample(frac=1,random_state=SEED).reset_index(drop=True)
print(data.shape);print(data.condition.value_counts().sort_index());data.head()

## Part 5 · What you just built

**In the notebook:** Build a balanced synthetic dataset and inspect class distributions.

**Takeaway:** Synthetic regimes should overlap realistically while remaining auditable.

◀ [Previous: The Seven Engine Measurements](https://engine-operating-condition.streamlit.app/?stage=inputs) &nbsp;|&nbsp; [Project overview](https://engine-operating-condition.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Checking and Scaling the Log](https://engine-operating-condition.streamlit.app/?stage=prepare) ▶

---
# 4. Checking and Scaling the Log
### Phase 2 of 6 · Preparing Sensor Records

## Part 1 · At the engine
Sensors can miss readings and RPM is numerically much larger than pressure or fuel flow.

## Part 2 · The engineering challenge
An MLP cannot accept missing values, and scaling on the full dataset leaks test information.

## Part 3 · Where the AI comes in
Split first, learn imputation and scaling parameters from training rows only, then transform validation and test rows.

**Mechanical Engineering:** Checking and Scaling the Log → **AI:** Cleaning and Normalization → `impute training median; StandardScaler`

> 🎬 **See this illustrated and interactive:** [https://engine-operating-condition.streamlit.app/?stage=prepare](https://engine-operating-condition.streamlit.app/?stage=prepare)

## Part 4 · The technical explanation

In [ ]:
# Add a few realistic missing sensor readings for the cleaning demonstration.
rng=np.random.default_rng(9);dirty=data.copy();dirty.loc[rng.choice(dirty.index,30,replace=False),"intake_c"]=np.nan
train,temp=train_test_split(dirty,test_size=.30,stratify=dirty.condition,random_state=SEED);val,test=train_test_split(temp,test_size=.50,stratify=temp.condition,random_state=SEED)
imputer=SimpleImputer(strategy="median").fit(train[FEATURES]);scaler=StandardScaler().fit(imputer.transform(train[FEATURES]))
def prep(d):return scaler.transform(imputer.transform(d[FEATURES]))
X_train,X_val,X_test=prep(train),prep(val),prep(test);y_train,y_val,y_test=train.condition.to_numpy(),val.condition.to_numpy(),test.condition.to_numpy()
print(X_train.shape,X_val.shape,X_test.shape)

## Part 5 · What you just built

**In the notebook:** Handle missing data and standardize seven inputs without leakage.

**Takeaway:** Preprocessing parameters are learned components of the deployed model.

◀ [Previous: Engine Operating Regimes](https://engine-operating-condition.streamlit.app/?stage=data) &nbsp;|&nbsp; [Project overview](https://engine-operating-condition.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Choosing the Right Neural Network](https://engine-operating-condition.streamlit.app/?stage=mlp) ▶

---
# 5. Choosing the Right Neural Network
### Phase 3 of 6 · Choosing the Network

## Part 1 · At the engine
The question concerns one synchronized sensor snapshot, not an image, waveform, or long history.

## Part 2 · The engineering challenge
Using a more fashionable architecture adds complexity without matching the structure of the data.

## Part 3 · Where the AI comes in
Dense layers are appropriate because every feature can interact with every other feature in the same row.

**Mechanical Engineering:** Choosing the Right Neural Network → **AI:** Multilayer Perceptron → `7 -> Dense32 -> Dense16 -> Softmax5`

> 🎬 **See this illustrated and interactive:** [https://engine-operating-condition.streamlit.app/?stage=mlp](https://engine-operating-condition.streamlit.app/?stage=mlp)

## Part 4 · The technical explanation

In [ ]:
model=Sequential([Input((7,)),Dense(32,activation="relu"),Dropout(.12),Dense(16,activation="relu"),Dense(5,activation="softmax")]);model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"]);model.summary()
print("Why MLP: one independent tabular row. CNN expects local signal/image structure; LSTM expects an ordered sequence; Transformer adds unnecessary capacity here.")

## Part 5 · What you just built

**In the notebook:** Build a compact MLP and contrast its input assumptions with CNN and LSTM models.

**Takeaway:** Architecture should follow data structure, not fashion.

◀ [Previous: Checking and Scaling the Log](https://engine-operating-condition.streamlit.app/?stage=prepare) &nbsp;|&nbsp; [Project overview](https://engine-operating-condition.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Learning Combined Operating Patterns](https://engine-operating-condition.streamlit.app/?stage=training) ▶

---
# 6. Learning Combined Operating Patterns
### Phase 4 of 6 · Learning Conditions

## Part 1 · At the engine
Each labelled test records the engine condition that produced its sensor combination.

## Part 2 · The engineering challenge
Class imbalance or repeated data can make accuracy misleading, especially for the broad Abnormal category.

## Part 3 · Where the AI comes in
Use balanced classes, stratified splits, validation monitoring, and class-specific test metrics.

**Mechanical Engineering:** Learning Combined Operating Patterns → **AI:** Supervised Classification → `cross-entropy + Adam + early stopping`

> 🎬 **See this illustrated and interactive:** [https://engine-operating-condition.streamlit.app/?stage=training](https://engine-operating-condition.streamlit.app/?stage=training)

## Part 4 · The technical explanation

In [ ]:
early=EarlyStopping(monitor="val_loss",patience=7,restore_best_weights=True);history=model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=60,batch_size=48,callbacks=[early],verbose=0)
fig,ax=plt.subplots(1,2,figsize=(12,4));ax[0].plot(history.history["loss"],label="train");ax[0].plot(history.history["val_loss"],label="validation");ax[1].plot(history.history["accuracy"],label="train");ax[1].plot(history.history["val_accuracy"],label="validation")
for a in ax:a.set_xlabel("Epoch");a.grid(alpha=.2);a.legend()
plt.show()

## Part 5 · What you just built

**In the notebook:** Train the MLP and plot loss and accuracy.

**Takeaway:** The test set must represent operating rows the network did not optimize against.

◀ [Previous: Choosing the Right Neural Network](https://engine-operating-condition.streamlit.app/?stage=mlp) &nbsp;|&nbsp; [Project overview](https://engine-operating-condition.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Current Engine Condition](https://engine-operating-condition.streamlit.app/?stage=prediction) ▶

---
# 7. Current Engine Condition
### Phase 5 of 6 · Engineering Response

## Part 1 · At the engine
The engineer needs a concise condition and an appropriate next inspection or load-management response.

## Part 2 · The engineering challenge
The largest probability can still be uncertain, and a classroom label must not directly control machinery.

## Part 3 · Where the AI comes in
Display all probabilities, identify the highest, and attach a transparent simulated recommendation outside the MLP.

**Mechanical Engineering:** Current Engine Condition → **AI:** Softmax Probabilities → `five classes + confidence + simulated action`

> 🎬 **See this illustrated and interactive:** [https://engine-operating-condition.streamlit.app/?stage=prediction](https://engine-operating-condition.streamlit.app/?stage=prediction)

## Part 4 · The technical explanation

In [ ]:
row=pd.DataFrame([example]);probs=model.predict(prep(row),verbose=0)[0];pred=int(np.argmax(probs));actions={0:"Continue simulated operation",1:"Simulated response: reduce load",2:"Simulated response: reduce load / inspect cooling",3:"Simulated response: inspect fuel-air condition",4:"Simulated response: schedule inspection"}
print("Predicted condition:",CLASSES[pred]);print("Confidence:",f"{probs[pred]:.1%}");print(actions[pred])
for c,p in zip(CLASSES,probs):print(f"{c:12s} {p:.1%}")

## Part 5 · What you just built

**In the notebook:** Predict an entered sensor row and print condition, confidence, and simulated response.

**Takeaway:** The MLP classifies; the recommendation layer remains visible and reviewable.

◀ [Previous: Learning Combined Operating Patterns](https://engine-operating-condition.streamlit.app/?stage=training) &nbsp;|&nbsp; [Project overview](https://engine-operating-condition.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: The Engine Monitoring Audit](https://engine-operating-condition.streamlit.app/?stage=audit) ▶

---
# 8. The Engine Monitoring Audit
### Phase 6 of 6 · Model Audit

## Part 1 · At the engine
Confusing High Load with Overheating has a different consequence from confusing Normal with Abnormal.

## Part 2 · The engineering challenge
Overall accuracy hides class-specific misses and overconfident predictions on unfamiliar sensor combinations.

## Part 3 · Where the AI comes in
Inspect the confusion matrix, per-class recall, calibration-style confidence bins, and out-of-range warnings.

**Mechanical Engineering:** The Engine Monitoring Audit → **AI:** Confusion Matrix and Confidence Review → `accuracy, per-class recall, uncertain cases`

> 🎬 **See this illustrated and interactive:** [https://engine-operating-condition.streamlit.app/?stage=audit](https://engine-operating-condition.streamlit.app/?stage=audit)

## Part 4 · The technical explanation

In [ ]:
probs=model.predict(X_test,verbose=0);pred=np.argmax(probs,axis=1);print(classification_report(y_test,pred,target_names=CLASSES));ConfusionMatrixDisplay(confusion_matrix(y_test,pred),display_labels=CLASSES).plot(cmap="Blues",xticks_rotation=25);plt.show()
confidence=probs.max(1);print("Low-confidence test rows (<60%):",int((confidence<.6).sum()))
mins=train[FEATURES].min();maxs=train[FEATURES].max();outside=((row[FEATURES]<mins)|(row[FEATURES]>maxs)).any(axis=1).iloc[0];print("Example outside training ranges:",outside)
print("Limitations: synthetic regimes, snapshot only, no transient detection, no engine-family transfer study, no calibrated decision costs, no direct machinery control.")

## Part 5 · What you just built

**In the notebook:** Report test metrics and state synthetic-data and deployment limitations.

**Takeaway:** A useful classifier must reveal where it is uncertain and where its training domain ends.

◀ [Previous: Current Engine Condition](https://engine-operating-condition.streamlit.app/?stage=prediction) &nbsp;|&nbsp; [Project overview](https://engine-operating-condition.streamlit.app/?stage=start)

---
# Final engineering conclusion

An MLP is appropriate because each example is one independent row of seven named sensor readings. The model returns five condition probabilities, while the recommendation remains a separate, transparent simulated layer.